In [ ]:
# CELL 1: Install
%%capture install_out
!pip install vllm aiohttp nest_asyncio tenacity pydantic

import subprocess
print("Install done.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

In [ ]:
# CELL 2: Upload DB file + fix Colab async
import nest_asyncio
nest_asyncio.apply()

from google.colab import files
print("Upload your .db file:")
uploaded = files.upload()

# After upload, find the DB path
import os
db_files = [f for f in uploaded.keys() if f.endswith('.db')]
DB_PATH = f"/content/{db_files[0]}"
print(f"DB loaded at: {DB_PATH}")

# Quick verify
import sqlite3
conn = sqlite3.connect(DB_PATH)
q_count = conn.execute("SELECT COUNT(*) FROM queries").fetchone()[0]
c_count = conn.execute("SELECT COUNT(*) FROM claims").fetchone()[0]
ao_count = conn.execute("SELECT COUNT(*) FROM agent_outputs").fetchone()[0]
conn.close()
print(f"queries={q_count}  claims={c_count}  agent_outputs={ao_count} (should be 0)")

Upload your .db file:


Saving mad_before_phase1_5090_ragfix_01_.db to mad_before_phase1_5090_ragfix_01_.db
DB loaded at: /content/mad_before_phase1_5090_ragfix_01_.db
queries=50  claims=414  agent_outputs=0 (should be 0)


In [ ]:
# CELL 3v2: Config — MAD v2

MODEL_NAME   = "Qwen/Qwen2.5-14B-Instruct-AWQ"
VLLM_PORT    = 8001
VLLM_URL     = f"http://localhost:{VLLM_PORT}/v1"
SERVED_NAME  = "agents"

MAX_CHUNK_CHARS  = 800
MAX_CLAIM_CHARS  = 300
MAX_PEER_CHARS   = 600   # slightly longer — agents need full peer context to disagree

# v2: different temps per round — Round 1 needs more diversity to break consensus
TEMPERATURES_R0 = {"agent_a": 0.3, "agent_b": 0.8, "agent_c": 0.5}
TEMPERATURES_R1 = {"agent_a": 0.4, "agent_b": 0.9, "agent_c": 0.5}

CLAIM_CONCURRENCY = 4
MAX_TOKENS_OUT    = 600   # more room for reasoning
VLLM_TIMEOUT      = 150

# v2: retry config
MAX_PARSE_RETRIES = 3
RETRY_BACKOFF     = [2, 5, 10]  # seconds between retries

# v2: confidence clamping (prevent overconfidence destroying GRPO signal)
CONF_MAX = 0.95
CONF_MIN = 0.05

print("✓ MAD v2 config set")

✓ MAD v2 config set


In [ ]:
# CELL 4v2: Launch vLLM — key change: --generation-config vllm
# This disables the model's baked-in generation_config.json defaults
# so OUR per-request temperatures are actually applied

import subprocess

LOG_PATH = "/content/vllm_server_v2.log"

vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model",                   MODEL_NAME,
    "--quantization",            "awq_marlin",   # faster than awq
    "--max-model-len",           "6144",
    "--gpu-memory-utilization",  "0.88",
    "--max-num-seqs",            "12",
    "--enable-prefix-caching",
    "--port",                    str(VLLM_PORT),
    "--served-model-name",       SERVED_NAME,
    "--trust-remote-code",
    "--dtype",                   "float16",
    "--generation-config",       "vllm",   # KEY: disables model's temp override
]

server_log  = open(LOG_PATH, "w")
server_proc = subprocess.Popen(vllm_cmd, stdout=server_log, stderr=subprocess.STDOUT)
print(f"Server PID: {server_proc.pid}  |  Log: {LOG_PATH}")
print("Run CELL 5 to wait for ready...")

Server PID: 2887  |  Log: /content/vllm_server_v2.log
Run CELL 5 to wait for ready...


In [ ]:
# CELL 5: Poll until vLLM is ready (model load takes 2-4 min on A100)
import time, requests

def wait_for_vllm(url: str, timeout_secs: int = 300):
    print("Waiting for vLLM server", end="")
    for i in range(timeout_secs // 5):
        try:
            r = requests.get(f"{url}/models", timeout=3)
            if r.status_code == 200:
                models = r.json().get("data", [])
                print(f"\n✓ Server ready! Models: {[m['id'] for m in models]}")
                return True
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(5)
    print("\n✗ Server failed to start — check logs below")
    return False

ready = wait_for_vllm(VLLM_URL)

if not ready:
    # Print last 30 lines of log to diagnose
    !tail -30 /content/vllm_server.log

Waiting for vLLM server..............................
✓ Server ready! Models: ['agents']


In [ ]:
# CELL 6: Data schemas — the contract between agents, DB, and judge

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from enum import Enum

class AgentVerdict(str, Enum):
    SUPPORTED     = "SUPPORTED"
    PARTIAL       = "PARTIAL"
    NOT_SUPPORTED = "NOT_SUPPORTED"
    IDK           = "IDK"

class AgentRole(str, Enum):
    AGENT_A = "agent_a"   # Verifier
    AGENT_B = "agent_b"   # Adversarial Auditor
    AGENT_C = "agent_c"   # Calibrator

class EvidenceCitation(BaseModel):
    chunk_id:       str
    relevant_quote: str

class AgentOutputFull(BaseModel):
    """Full output → written to SQLite. NEVER passed to peers or judge."""
    agent_role:          AgentRole
    round_num:           int
    verdict:             AgentVerdict
    reasoning:           str
    evidence_cited:      List[EvidenceCitation]
    confidence_internal: float = Field(ge=0.0, le=1.0)

class AgentOutputStripped(BaseModel):
    """Stripped output → safe to pass to peer agents. No confidence. No role attribution."""
    debater_label:  str        # "Debater 1", "Debater 2", "Debater 3"
    round_num:      int
    verdict:        AgentVerdict
    reasoning:      str        # may be truncated to MAX_PEER_CHARS
    evidence_cited: List[EvidenceCitation]

def strip_for_peer(output: AgentOutputFull, debater_label: str,
                   max_reasoning_chars: int = MAX_PEER_CHARS) -> AgentOutputStripped:
    """
    THE ONLY function that creates AgentOutputStripped.
    Removes confidence_internal and anonymizes agent identity.
    This is the bias control choke point — never bypass it.
    """
    return AgentOutputStripped(
        debater_label  = debater_label,
        round_num      = output.round_num,
        verdict        = output.verdict,
        reasoning      = output.reasoning[:max_reasoning_chars],
        evidence_cited = output.evidence_cited,
    )

# Verify bias control works
def _test_strip():
    full = AgentOutputFull(
        agent_role=AgentRole.AGENT_A, round_num=0,
        verdict=AgentVerdict.SUPPORTED, reasoning="test",
        evidence_cited=[], confidence_internal=0.85
    )
    stripped = strip_for_peer(full, "Debater 1")
    s = stripped.model_dump_json()
    assert "confidence" not in s.lower(), "LEAK: confidence in stripped output!"
    assert "agent_a"   not in s.lower(), "LEAK: agent role in stripped output!"
    assert "agent_b"   not in s.lower(), "LEAK: agent role in stripped output!"
    print("✓ strip_for_peer bias control test passed")

_test_strip()

✓ strip_for_peer bias control test passed


In [ ]:
# CELL 7v2: System prompts — v2 changes:
# 1. Explicitly penalize PARTIAL as default hedge
# 2. Agent A pushed harder toward SUPPORTED when evidence warrants
# 3. Agent B pushed harder toward NOT_SUPPORTED
# 4. Confidence 1.0 explicitly forbidden

AGENT_A_SYSTEM = """You are a strict regulatory compliance verifier in a multi-agent debate.

YOUR ROLE: Find the strongest case FOR the claim being correct, based on the evidence.

RULES:
1. Analyze the evidence. Find what SUPPORTS the claim.
2. Verdict:
   - SUPPORTED: evidence clearly and directly backs the claim → USE THIS when evidence supports
   - PARTIAL: only when evidence supports SOME but not all parts of the claim → be specific about what's missing
   - NOT_SUPPORTED: evidence contradicts or is completely silent on the claim
   - IDK: evidence is genuinely too ambiguous — use rarely
3. WARNING: Do NOT default to PARTIAL as a hedge. If the evidence supports the claim, say SUPPORTED.
   PARTIAL requires you to explain EXACTLY which part is supported and which is not.
4. Cite specific chunk_ids. Quotes must be actual text from the chunks.
5. confidence_internal: your true certainty. NEVER assign 1.0 — there is always uncertainty.
   NEVER assign 0.0 unless you have zero basis to decide. Range: 0.05 to 0.95.

OUTPUT FORMAT: Valid JSON only. No markdown fences. No text outside JSON.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Specific analysis referencing chunk content...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "exact text from chunk"}],
    "confidence_internal": 0.05-0.95
}"""

AGENT_B_SYSTEM = """You are an adversarial auditor in a multi-agent regulatory compliance debate.

YOUR ROLE: Find what is WRONG, UNSUPPORTED, or MISLEADING about the claim.

RULES:
1. Analyze the evidence. Find what UNDERMINES the claim.
2. Verdict:
   - NOT_SUPPORTED: evidence contradicts the claim or simply doesn't back it → USE THIS when claim is unsupported
   - PARTIAL: only when part of the claim is wrong or missing — specify exactly which part
   - SUPPORTED: if the claim is genuinely well-supported despite your scrutiny — be honest
   - IDK: only when evidence is truly ambiguous
3. WARNING: Do NOT default to PARTIAL as a safe middle ground. If the claim is wrong or unsupported, say NOT_SUPPORTED.
   If the evidence is silent on the claim, that is NOT_SUPPORTED, not PARTIAL.
4. Look specifically for: scope errors, missing exceptions, unsupported specifics, hallucinated references.
5. Cite chunk_ids that FAIL to support or contradict the claim.
6. confidence_internal: NEVER assign 1.0 or 0.0. Range: 0.05 to 0.95.

OUTPUT FORMAT: Valid JSON only. No markdown fences. No text outside JSON.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Specific challenge referencing chunk content...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "exact text from chunk"}],
    "confidence_internal": 0.05-0.95
}"""

AGENT_C_SYSTEM = """You are a neutral evidence calibrator in a multi-agent regulatory compliance debate.

YOUR ROLE: Weigh evidence honestly on BOTH sides. No prior stance.

RULES:
1. Analyze evidence for AND against the claim separately.
2. Verdict:
   - SUPPORTED: evidence clearly favors the claim → say so, do not hedge
   - NOT_SUPPORTED: evidence clearly goes against the claim → say so
   - PARTIAL: ONLY when evidence genuinely cuts both ways — explain exactly what is supported and what is not
   - IDK: when evidence is truly insufficient to decide
3. WARNING: PARTIAL is not the default safe answer. If evidence clearly points one way, commit to SUPPORTED or NOT_SUPPORTED.
   Intellectual honesty means calling clear cases clearly.
4. Cite chunk_ids for BOTH directions — supporting and undermining evidence.
5. confidence_internal: calibrated 0.05–0.95. 0.5 means genuinely 50/50. Not a default.

OUTPUT FORMAT: Valid JSON only. No markdown fences. No text outside JSON.
{
    "verdict": "SUPPORTED | PARTIAL | NOT_SUPPORTED | IDK",
    "reasoning": "Balanced analysis with evidence for and against...",
    "evidence_cited": [{"chunk_id": "...", "relevant_quote": "exact text from chunk"}],
    "confidence_internal": 0.05-0.95
}"""

SYSTEM_PROMPTS = {
    "agent_a": AGENT_A_SYSTEM,
    "agent_b": AGENT_B_SYSTEM,
    "agent_c": AGENT_C_SYSTEM,
}
print("✓ v2 system prompts defined")

✓ v2 system prompts defined


In [ ]:
# CELL 8v2: DB helpers — added: per-run stats table, parse_failed flag

import sqlite3, json, threading, uuid, logging
from datetime import datetime

db_lock = threading.Lock()
logger  = logging.getLogger("MAD.db")

# v2: add parse_failed column to track quality
SCHEMA_PATCH = """
ALTER TABLE agent_outputs ADD COLUMN parse_failed INTEGER DEFAULT 0;
ALTER TABLE agent_outputs ADD COLUMN call_failed  INTEGER DEFAULT 0;
ALTER TABLE agent_outputs ADD COLUMN retry_count  INTEGER DEFAULT 0;
"""

def apply_schema_patch():
    conn = sqlite3.connect(DB_PATH)
    for stmt in SCHEMA_PATCH.strip().split(";"):
        stmt = stmt.strip()
        if not stmt:
            continue
        try:
            conn.execute(stmt)
        except sqlite3.OperationalError as e:
            if "duplicate column" in str(e).lower():
                pass  # already applied
            else:
                logger.warning(f"Schema patch: {e}")
    conn.commit()
    conn.close()
    print("✓ Schema patch applied")

apply_schema_patch()

def write_agent_output(
    claim_id, agent_role, round_num,
    verdict, reasoning, evidence_cited,
    confidence_internal, raw_response,
    latency_ms, tokens_in, tokens_out,
    parse_failed=False, call_failed=False, retry_count=0
):
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO agent_outputs
            (output_id, claim_id, agent_role, round_num, verdict, reasoning,
             evidence_cited, confidence_internal, raw_response,
             latency_ms, tokens_in, tokens_out, timestamp,
             parse_failed, call_failed, retry_count)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,CURRENT_TIMESTAMP,?,?,?)
        """, (
            str(uuid.uuid4()), claim_id, agent_role, round_num,
            verdict, reasoning, json.dumps([e.model_dump() if hasattr(e,'model_dump') else e for e in evidence_cited]),
            confidence_internal, raw_response[:4000],  # cap raw to prevent bloat
            latency_ms, tokens_in, tokens_out,
            int(parse_failed), int(call_failed), retry_count
        ))
        conn.commit()
        conn.close()

def write_agent_delta(claim_id, agent_role, conf_r0, conf_r1, verdict_r0, verdict_r1):
    with db_lock:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("""
            INSERT OR REPLACE INTO agent_deltas
            (delta_id, claim_id, agent_role, confidence_r0, confidence_r1,
             delta, verdict_r0, verdict_r1, verdict_changed)
            VALUES (?,?,?,?,?,?,?,?,?)
        """, (
            str(uuid.uuid4()), claim_id, agent_role,
            conf_r0, conf_r1, round(conf_r1 - conf_r0, 4),
            verdict_r0, verdict_r1, int(verdict_r0 != verdict_r1)
        ))
        conn.commit()
        conn.close()

# v2: run-level stats logger
class RunStats:
    def __init__(self):
        self.parse_fails = 0
        self.call_fails  = 0
        self.total_calls = 0
        self.verdict_counts = {}
        self.r0_consensus   = 0
        self.r0_total       = 0

    def record(self, parsed, role, round_num):
        self.total_calls += 1
        if parsed.get("_parse_failed"): self.parse_fails += 1
        if parsed.get("_call_failed"):  self.call_fails += 1
        v = parsed.get("verdict", "IDK")
        self.verdict_counts[v] = self.verdict_counts.get(v, 0) + 1

    def summary(self):
        total = self.total_calls or 1
        print(f"\n=== Run Stats ===")
        print(f"  Total calls:   {self.total_calls}")
        print(f"  Parse fails:   {self.parse_fails} ({self.parse_fails/total*100:.1f}%)")
        print(f"  Call fails:    {self.call_fails}  ({self.call_fails/total*100:.1f}%)")
        print(f"  Verdict dist:  {self.verdict_counts}")

run_stats = RunStats()
print("✓ DB helpers v2 ready")

✓ Schema patch applied
✓ DB helpers v2 ready


In [ ]:
# CELL 9v2: vLLM client — retries, better parse, confidence clamping

import aiohttp, asyncio, time, re, json, logging

logger = logging.getLogger("MAD.v2")

def parse_agent_json(raw: str) -> dict:
    raw = raw.strip()
    for attempt_fn in [
        lambda r: json.loads(r),
        lambda r: json.loads(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', r, re.DOTALL).group(1)),
        lambda r: json.loads(re.search(r'\{.*\}', r, re.DOTALL).group(0)),
    ]:
        try:
            result = attempt_fn(raw)
            if isinstance(result, dict) and "verdict" in result:
                return result
        except Exception:
            continue

    # Field-level regex fallback
    verdict_m = re.search(r'"verdict"\s*:\s*"([^"]+)"', raw)
    conf_m    = re.search(r'"confidence_internal"\s*:\s*([0-9.]+)', raw)
    reason_m  = re.search(r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*)"', raw)
    return {
        "verdict":             verdict_m.group(1) if verdict_m else "IDK",
        "reasoning":           reason_m.group(1)  if reason_m  else f"[PARSE_FAILED] {raw[:400]}",
        "evidence_cited":      [],
        "confidence_internal": float(conf_m.group(1)) if conf_m else 0.5,
        "_parse_failed":       True,
    }

def normalize_verdict(v: str) -> str:
    return {
        "SUPPORTED": "SUPPORTED", "SUPPORT": "SUPPORTED",
        "NOT_SUPPORTED": "NOT_SUPPORTED", "NOTSUPPORTED": "NOT_SUPPORTED",
        "NOT SUPPORTED": "NOT_SUPPORTED", "UNSUPPORTED": "NOT_SUPPORTED",
        "PARTIAL": "PARTIAL", "PARTIALLY SUPPORTED": "PARTIAL",
        "IDK": "IDK", "INSUFFICIENT": "IDK", "UNKNOWN": "IDK",
    }.get(v.upper().strip(), "IDK")

def clamp_confidence(c: float) -> float:
    """Prevent 0.0 and 1.0 which destroy GRPO Brier signal."""
    try:
        return max(CONF_MIN, min(CONF_MAX, float(c)))
    except Exception:
        return 0.5

def safe_evidence(raw_list: list) -> list:
    result = []
    for e in raw_list:
        if not isinstance(e, dict) or "chunk_id" not in e:
            continue
        result.append(EvidenceCitation(
            chunk_id=str(e.get("chunk_id", "unknown"))[:100],
            relevant_quote=str(e.get("relevant_quote", e.get("quote", "")))[:300],
        ))
    return result

async def call_agent_with_retry(
    session: aiohttp.ClientSession,
    system_prompt: str,
    user_prompt: str,
    temperature: float,
    role: str,
    claim_id: str,
    round_num: int,
) -> tuple[dict, str, int, int, int]:
    """
    Call vLLM with exponential backoff retries.
    On parse failure: retry with explicit JSON reminder appended.
    Returns: (parsed, raw_text, latency_ms, tokens_in, tokens_out)
    """
    last_error = None

    for attempt in range(MAX_PARSE_RETRIES):
        # On retry for parse failure, append a reminder
        prompt = user_prompt
        if attempt > 0 and last_error == "parse":
            prompt += "\n\nREMINDER: Output ONLY valid JSON. Start with { and end with }. No other text."

        payload = {
            "model":       SERVED_NAME,
            "messages":    [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt},
            ],
            "temperature": temperature,
            "max_tokens":  MAX_TOKENS_OUT,
            "stream":      False,
        }

        start = time.time()
        try:
            async with session.post(
                f"{VLLM_URL}/chat/completions",
                json=payload,
                timeout=aiohttp.ClientTimeout(total=VLLM_TIMEOUT),
            ) as resp:
                resp.raise_for_status()
                data       = await resp.json()
                latency_ms = int((time.time() - start) * 1000)
                raw_text   = data["choices"][0]["message"]["content"]
                tokens_in  = data["usage"]["prompt_tokens"]
                tokens_out = data["usage"]["completion_tokens"]

        except Exception as e:
            latency_ms = int((time.time() - start) * 1000)
            logger.warning(f"[{role}] call error attempt {attempt+1}/{MAX_PARSE_RETRIES}: {e}")
            if attempt < MAX_PARSE_RETRIES - 1:
                await asyncio.sleep(RETRY_BACKOFF[attempt])
                last_error = "call"
                continue
            raw_text  = f"[CALL_ERROR] {str(e)}"
            tokens_in = tokens_out = 0
            return (
                {"verdict": "IDK", "reasoning": raw_text,
                 "evidence_cited": [], "confidence_internal": 0.5, "_call_failed": True},
                raw_text, latency_ms, 0, 0
            )

        parsed = parse_agent_json(raw_text)

        # Check if parse succeeded
        if parsed.get("_parse_failed"):
            logger.warning(
                f"[{role}] R{round_num} parse fail attempt {attempt+1}/{MAX_PARSE_RETRIES} "
                f"| claim={claim_id[:8]} | raw={raw_text[:80]}"
            )
            if attempt < MAX_PARSE_RETRIES - 1:
                await asyncio.sleep(RETRY_BACKOFF[attempt])
                last_error = "parse"
                continue
            # Final attempt still failed — use fallback with logged warning
            logger.error(f"[{role}] R{round_num} all retries exhausted | claim={claim_id[:8]}")
        else:
            if attempt > 0:
                logger.info(f"[{role}] R{round_num} recovered on attempt {attempt+1}")

        # Normalize and clamp
        parsed["verdict"]             = normalize_verdict(parsed.get("verdict", "IDK"))
        parsed["confidence_internal"] = clamp_confidence(parsed.get("confidence_internal", 0.5))
        parsed["evidence_cited"]      = safe_evidence(parsed.get("evidence_cited", []))

        return parsed, raw_text, latency_ms, tokens_in, tokens_out

    # Should never reach here
    return ({"verdict": "IDK", "reasoning": "[EXHAUSTED]", "evidence_cited": [],
             "confidence_internal": 0.5}, "[EXHAUSTED]", 0, 0, 0)

print("✓ v2 client with retries ready")

✓ v2 client with retries ready


In [ ]:
# CELL 10: Build user prompts for Round 0 and Round 1

def format_chunks(chunks: list, max_chars: int = MAX_CHUNK_CHARS) -> str:
    """Format RAG chunks for agent prompts. Truncates each chunk text."""
    parts = []
    for i, c in enumerate(chunks):
        text = c["text"][:max_chars]
        if len(c["text"]) > max_chars:
            text += "... [truncated]"
        parts.append(
            f"[Chunk {i+1} | ID: {c['chunk_id']} | Source: {c['source_file']}]\n{text}"
        )
    return "\n\n".join(parts)

def build_round0_prompt(query: str, claim_text: str, chunks: list) -> str:
    claim_text = claim_text[:MAX_CLAIM_CHARS]
    return f"""USER QUERY: {query}

CLAIM TO VERIFY:
{claim_text}

RETRIEVED EVIDENCE:
{format_chunks(chunks)}

Provide your independent verdict on this claim."""

def build_round1_prompt(
    query: str,
    claim_text: str,
    chunks: list,
    peer1: AgentOutputStripped,
    peer2: AgentOutputStripped,
) -> str:
    claim_text = claim_text[:MAX_CLAIM_CHARS]

    def fmt_peer(p: AgentOutputStripped) -> str:
        evidence_str = ", ".join(e.chunk_id for e in p.evidence_cited[:3]) or "none"
        return (
            f"Verdict: {p.verdict}\n"
            f"Reasoning: {p.reasoning[:MAX_PEER_CHARS]}\n"
            f"Evidence cited: {evidence_str}"
        )

    return f"""USER QUERY: {query}

CLAIM TO VERIFY:
{claim_text}

RETRIEVED EVIDENCE:
{format_chunks(chunks)}

OTHER DEBATERS' POSITIONS:

[{peer1.debater_label}]
{fmt_peer(peer1)}

[{peer2.debater_label}]
{fmt_peer(peer2)}

Review the positions above carefully. You may update or maintain your Round 0 verdict.
If a debater made a valid point supported by evidence, acknowledge it.
If their argument is weak or unsupported, push back with evidence.
Provide your final verdict."""

print("✓ Prompt builders ready")

# Show worst-case token estimate
import json
sample_q = queries[22]  # q_023 — largest chunks
sample_c = load_claims_for_query(sample_q["query_id"])[0]
r0_prompt = build_round0_prompt(sample_q["user_query"], sample_c["claim_text"], sample_q["rag_chunks"])
print(f"Worst-case R0 prompt: {len(r0_prompt)} chars ≈ {len(r0_prompt)//4} tokens")
print(f"System prompt: ≈ {len(AGENT_A_SYSTEM)//4} tokens")
print(f"Estimated total input: ≈ {(len(r0_prompt)+len(AGENT_A_SYSTEM))//4} tokens (limit=6144)")

✓ Prompt builders ready


NameError: name 'queries' is not defined

In [ ]:
# CELL 11: Round 0 — all three agents independently assess each claim

import asyncio, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("MAD")

AGENT_ROLES   = ["agent_a", "agent_b", "agent_c"]
DEBATER_LABEL = {"agent_a": "Debater 1", "agent_b": "Debater 2", "agent_c": "Debater 3"}

async def debate_claim_round0(
    session:    aiohttp.ClientSession,
    query:      dict,
    claim:      dict,
) -> dict:
    """
    Fire all 3 agents simultaneously on one claim.
    Returns: {agent_role: AgentOutputFull}
    """
    tasks = {}
    for role in AGENT_ROLES:
        prompt = build_round0_prompt(
            query["user_query"],
            claim["claim_text"],
            query["rag_chunks"],
        )
        tasks[role] = asyncio.create_task(
            call_agent_async(
                session,
                SYSTEM_PROMPTS[role],
                prompt,
                TEMPERATURES[role],
            )
        )

    results = {}
    for role, task in tasks.items():
        parsed, raw, lat, ti, to = await task

        # Build full output object
        evidence_cited = [
            EvidenceCitation(**e) if isinstance(e, dict) else e
            for e in parsed.get("evidence_cited", [])
        ]
        output = AgentOutputFull(
            agent_role=AgentRole(role),
            round_num=0,
            verdict=AgentVerdict(parsed["verdict"]),
            reasoning=parsed.get("reasoning", ""),
            evidence_cited=evidence_cited,
            confidence_internal=parsed["confidence_internal"],
        )
        results[role] = output

        # Write to DB immediately
        write_agent_output(
            claim_id=claim["claim_id"],
            agent_role=role, round_num=0,
            verdict=output.verdict.value,
            reasoning=output.reasoning,
            evidence_cited=[e.model_dump() for e in output.evidence_cited],
            confidence_internal=output.confidence_internal,
            raw_response=raw,
            latency_ms=lat, tokens_in=ti, tokens_out=to,
        )

        logger.info(
            f"R0 | {claim['claim_id'][:8]} | {role} | "
            f"{output.verdict.value} | conf={output.confidence_internal:.2f} | {lat}ms"
        )

    return results

print("✓ Round 0 debate function ready")

In [ ]:
# CELL 12: Round 1 — each agent sees the other TWO agents' stripped R0 outputs

async def debate_claim_round1(
    session:    aiohttp.ClientSession,
    query:      dict,
    claim:      dict,
    r0_outputs: dict,   # {agent_role: AgentOutputFull}
) -> dict:
    """
    Round 1: each agent sees the other two agents' R0 (stripped — no confidence, no attribution).
    Returns: {agent_role: AgentOutputFull}
    """
    # Build stripped peer views for every agent
    # agent_a sees stripped(agent_b) + stripped(agent_c)
    # agent_b sees stripped(agent_a) + stripped(agent_c)
    # agent_c sees stripped(agent_a) + stripped(agent_b)

    stripped = {
        role: strip_for_peer(output, DEBATER_LABEL[role])
        for role, output in r0_outputs.items()
    }

    other_roles = {
        "agent_a": ("agent_b", "agent_c"),
        "agent_b": ("agent_a", "agent_c"),
        "agent_c": ("agent_a", "agent_b"),
    }

    tasks = {}
    for role in AGENT_ROLES:
        p1_role, p2_role = other_roles[role]
        prompt = build_round1_prompt(
            query["user_query"],
            claim["claim_text"],
            query["rag_chunks"],
            peer1=stripped[p1_role],
            peer2=stripped[p2_role],
        )
        tasks[role] = asyncio.create_task(
            call_agent_async(
                session,
                SYSTEM_PROMPTS[role],
                prompt,
                TEMPERATURES[role],
            )
        )

    results = {}
    for role, task in tasks.items():
        parsed, raw, lat, ti, to = await task

        evidence_cited = [
            EvidenceCitation(**e) if isinstance(e, dict) else e
            for e in parsed.get("evidence_cited", [])
        ]
        output = AgentOutputFull(
            agent_role=AgentRole(role),
            round_num=1,
            verdict=AgentVerdict(parsed["verdict"]),
            reasoning=parsed.get("reasoning", ""),
            evidence_cited=evidence_cited,
            confidence_internal=parsed["confidence_internal"],
        )
        results[role] = output

        # Write Round 1 output to DB
        write_agent_output(
            claim_id=claim["claim_id"],
            agent_role=role, round_num=1,
            verdict=output.verdict.value,
            reasoning=output.reasoning,
            evidence_cited=[e.model_dump() for e in output.evidence_cited],
            confidence_internal=output.confidence_internal,
            raw_response=raw,
            latency_ms=lat, tokens_in=ti, tokens_out=to,
        )

        # Write delta (R0 → R1)
        r0 = r0_outputs[role]
        write_agent_delta(
            claim_id=claim["claim_id"],
            agent_role=role,
            conf_r0=r0.confidence_internal,
            conf_r1=output.confidence_internal,
            verdict_r0=r0.verdict.value,
            verdict_r1=output.verdict.value,
        )

        logger.info(
            f"R1 | {claim['claim_id'][:8]} | {role} | "
            f"{r0.verdict.value}→{output.verdict.value} | "
            f"conf {r0.confidence_internal:.2f}→{output.confidence_internal:.2f} | "
            f"Δ={output.confidence_internal - r0.confidence_internal:+.2f} | {lat}ms"
        )

    return results

print("✓ Round 1 debate function ready")

In [ ]:
# CELL 13: Orchestrate full debate for one query (all its claims)

async def debate_query(
    session:    aiohttp.ClientSession,
    query:      dict,
    sem:        asyncio.Semaphore,
) -> dict:
    """
    Run full 2-round 3-agent debate for every claim in this query.
    Semaphore limits how many claims are debated concurrently.
    Returns: summary stats for this query.
    """
    claims = load_claims_for_query(query["query_id"])
    if not claims:
        logger.warning(f"No claims for {query['query_id']}, skipping")
        return {"query_id": query["query_id"], "claims": 0}

    async def debate_one_claim(claim: dict):
        async with sem:
            logger.info(f"→ Claim {claim['claim_index']} | {claim['claim_id'][:8]} | {claim['claim_text'][:60]}...")
            # Round 0
            r0 = await debate_claim_round0(session, query, claim)
            # Round 1
            r1 = await debate_claim_round1(session, query, claim, r0)
            return r0, r1

    tasks = [debate_one_claim(c) for c in claims]
    all_results = await asyncio.gather(*tasks, return_exceptions=True)

    errors = [r for r in all_results if isinstance(r, Exception)]
    if errors:
        logger.error(f"Query {query['query_id']} had {len(errors)} claim errors: {errors[:2]}")

    return {
        "query_id": query["query_id"],
        "claims":   len(claims),
        "errors":   len(errors),
    }

print("✓ Per-query orchestrator ready")

In [ ]:
# CELL 14: Main run — all 50 queries
# Skips queries that already have complete agent_outputs (safe to re-run)

async def run_all_queries():
    queries = load_all_queries()
    sem     = asyncio.Semaphore(CLAIM_CONCURRENCY)

    total_claims = sum(len(load_claims_for_query(q["query_id"])) for q in queries)
    print(f"Starting MAD debate: {len(queries)} queries, {total_claims} claims")
    print(f"3 agents × 2 rounds × {total_claims} claims = {total_claims*6} LLM calls total")
    print(f"Claim concurrency: {CLAIM_CONCURRENCY} (= {CLAIM_CONCURRENCY*3} concurrent agent calls)\n")

    start_time = time.time()
    summaries  = []

    async with aiohttp.ClientSession() as session:
        for i, query in enumerate(queries):
            qid = query["query_id"]

            if query_already_done(qid):
                print(f"[{i+1:02d}/50] {qid} — already done, skipping")
                continue

            claims = load_claims_for_query(qid)
            print(f"[{i+1:02d}/50] {qid} | {len(claims)} claims | {query['user_query'][:60]}...")

            try:
                summary = await debate_query(session, query, sem)
                summaries.append(summary)
            except Exception as e:
                logger.error(f"Query {qid} failed: {e}")
                summaries.append({"query_id": qid, "claims": 0, "errors": 1})

            # Progress
            elapsed = time.time() - start_time
            done_q  = i + 1
            rate    = done_q / elapsed
            eta     = (len(queries) - done_q) / rate if rate > 0 else 0
            print(f"    elapsed={elapsed/60:.1f}min | ETA={eta/60:.1f}min\n")

    total_elapsed = time.time() - start_time
    print(f"\n✓ All queries done in {total_elapsed/60:.1f} minutes")
    print(f"  Total summaries: {len(summaries)}")
    return summaries

# RUN IT
summaries = asyncio.run(run_all_queries())

In [ ]:
# CELL 15: Verify results and inspect debate quality

import sqlite3, json

conn = sqlite3.connect(DB_PATH)

# Row counts
print("=== TABLE COUNTS ===")
for table in ["queries", "claims", "agent_outputs", "agent_deltas"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

# Verdict distribution per agent
print("\n=== VERDICT DISTRIBUTION (Round 1) ===")
rows = conn.execute("""
    SELECT agent_role, verdict, COUNT(*) as n
    FROM agent_outputs
    WHERE round_num = 1
    GROUP BY agent_role, verdict
    ORDER BY agent_role, verdict
""").fetchall()
for r in rows:
    print(f"  {r[0]} | {r[1]:<15} | {r[2]}")

# Average confidence per agent per round
print("\n=== AVG CONFIDENCE PER AGENT PER ROUND ===")
rows = conn.execute("""
    SELECT agent_role, round_num, ROUND(AVG(confidence_internal),3) as avg_conf
    FROM agent_outputs
    GROUP BY agent_role, round_num
    ORDER BY agent_role, round_num
""").fetchall()
for r in rows:
    print(f"  {r[0]} R{r[1]} | avg_conf={r[2]}")

# Large confidence shifts (potential GRPO gold)
print("\n=== TOP 10 LARGEST CONFIDENCE DELTAS ===")
rows = conn.execute("""
    SELECT d.agent_role, d.claim_id, d.delta,
           d.verdict_r0, d.verdict_r1, d.verdict_changed,
           c.claim_text
    FROM agent_deltas d
    JOIN claims c USING (claim_id)
    ORDER BY ABS(d.delta) DESC
    LIMIT 10
""").fetchall()
for r in rows:
    changed = "✓ FLIPPED" if r[5] else ""
    print(f"  {r[0]} | Δ={r[2]:+.2f} | {r[3]}→{r[4]} {changed}")
    print(f"    claim: {r[6][:80]}")

# Disagreement between agents (judge-needed cases)
print("\n=== CLAIMS WHERE ALL 3 AGENTS DISAGREE (R1) ===")
rows = conn.execute("""
    SELECT claim_id, GROUP_CONCAT(agent_role||':'||verdict, ' | ') as positions
    FROM agent_outputs
    WHERE round_num = 1
    GROUP BY claim_id
    HAVING COUNT(DISTINCT verdict) = 3
    LIMIT 10
""").fetchall()
print(f"  Total 3-way disagreements: {len(rows)}")
for r in rows[:5]:
    print(f"  {r[0][:12]} | {r[1]}")

conn.close()

In [ ]:
# CELL 16: Download the updated DB back to your machine

from google.colab import files

print("Downloading updated DB...")
files.download(DB_PATH)
print("Done. DB now contains agent_outputs + agent_deltas for all claims.")
print("Next step: Stage 2 — run the Judge on this DB.")

In [ ]:
# CELL 17v2: Export MAD outputs → Unsloth-ready JSONL for GRPO

import json, sqlite3

def brier_reward(confidence: float, v_label: float) -> float:
    """Brier-based reward: R = 2*p*v - p^2. Range [-1, 1]."""
    p = confidence
    v = v_label
    return round(2 * p * v - p ** 2, 4)

def export_grpo_dataset(db_path: str, output_path: str):
    """
    Export debate data as GRPO training records for Unsloth.

    Each record = one (claim, agent, round) with:
      - prompt: what the agent saw (system + user)
      - completion: what the agent output
      - reward: Brier score after judge verdict (None if judge not run yet)
      - metadata: for filtering/debugging
    """
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    records = conn.execute("""
        SELECT
            ao.output_id,
            ao.claim_id,
            ao.agent_role,
            ao.round_num,
            ao.verdict,
            ao.reasoning,
            ao.confidence_internal,
            ao.raw_response,
            ao.parse_failed,
            ao.latency_ms,
            c.claim_text,
            c.is_material,
            c.is_critical,
            c.confidence_prior,
            q.user_query,
            q.rag_chunks,
            q.baseline_answer,
            q.query_id,
            jv.v_label,
            jv.judge_confidence,
            jv.judge_reasoning
        FROM agent_outputs ao
        JOIN claims c  USING (claim_id)
        JOIN queries q USING (query_id)
        LEFT JOIN judge_verdicts jv USING (claim_id)
        ORDER BY q.query_id, c.claim_index, ao.agent_role, ao.round_num
    """).fetchall()

    conn.close()

    grpo_records = []
    sft_records  = []
    dpo_pairs    = []

    # Group by (claim_id) to build DPO pairs across agents
    from collections import defaultdict
    by_claim = defaultdict(list)
    for r in records:
        by_claim[r["claim_id"]].append(r)

    for claim_id, claim_records in by_claim.items():
        # ── GRPO records (one per agent output) ──────────────────────────
        for r in claim_records:
            v_label = r["v_label"]  # None if judge not run

            reward = None
            if v_label is not None:
                reward = brier_reward(r["confidence_internal"], v_label)

            chunks = json.loads(r["rag_chunks"])
            chunks_text = "\n\n".join(
                f"[{c['chunk_id']}] {c['text'][:800]}" for c in chunks
            )

            # Build the prompt the agent saw
            if r["round_num"] == 0:
                user_prompt = (
                    f"USER QUERY: {r['user_query']}\n\n"
                    f"CLAIM TO VERIFY:\n{r['claim_text']}\n\n"
                    f"RETRIEVED EVIDENCE:\n{chunks_text}\n\n"
                    f"Provide your independent verdict on this claim."
                )
            else:
                user_prompt = (
                    f"USER QUERY: {r['user_query']}\n\n"
                    f"CLAIM TO VERIFY:\n{r['claim_text']}\n\n"
                    f"RETRIEVED EVIDENCE:\n{chunks_text}\n\n"
                    f"[Round 1 — peer views included in original run]\n\n"
                    f"Provide your final verdict."
                )

            grpo_records.append({
                "id":           r["output_id"],
                "query_id":     r["query_id"],
                "claim_id":     claim_id,
                "agent_role":   r["agent_role"],
                "round_num":    r["round_num"],
                "is_material":  bool(r["is_material"]),
                "is_critical":  bool(r["is_critical"]),
                # Prompt/completion pair
                "system":       SYSTEM_PROMPTS[r["agent_role"]],
                "prompt":       user_prompt,
                "completion":   r["raw_response"],
                # Parsed output
                "verdict":      r["verdict"],
                "confidence":   r["confidence_internal"],
                # Reward signal (None until Stage 2 runs)
                "v_label":      v_label,
                "brier_reward": reward,
                # Quality flags
                "parse_failed": bool(r["parse_failed"]),
                "latency_ms":   r["latency_ms"],
            })

            # ── SFT records (high-quality outputs only) ───────────────────
            if (not r["parse_failed"]
                    and reward is not None
                    and reward > 0.5          # agent was right and confident
                    and r["round_num"] == 1): # final round reasoning is richer
                sft_records.append({
                    "system":     SYSTEM_PROMPTS[r["agent_role"]],
                    "prompt":     user_prompt,
                    "completion": r["raw_response"],
                    "metadata": {
                        "claim_id":   claim_id,
                        "agent_role": r["agent_role"],
                        "verdict":    r["verdict"],
                        "confidence": r["confidence_internal"],
                        "reward":     reward,
                    }
                })

        # ── DPO pairs (best vs worst agent on same claim, R1 only) ────────
        r1_rows = [r for r in claim_records
                   if r["round_num"] == 1
                   and r["v_label"] is not None
                   and not r["parse_failed"]]

        if len(r1_rows) >= 2:
            scored = sorted(
                r1_rows,
                key=lambda r: brier_reward(r["confidence_internal"], r["v_label"]),
                reverse=True
            )
            winner, loser = scored[0], scored[-1]

            if brier_reward(winner["confidence_internal"], winner["v_label"]) > \
               brier_reward(loser["confidence_internal"],  loser["v_label"]) + 0.1:

                chunks      = json.loads(winner["rag_chunks"])
                chunks_text = "\n\n".join(
                    f"[{c['chunk_id']}] {c['text'][:800]}" for c in chunks
                )
                shared_prompt = (
                    f"USER QUERY: {winner['user_query']}\n\n"
                    f"CLAIM TO VERIFY:\n{winner['claim_text']}\n\n"
                    f"RETRIEVED EVIDENCE:\n{chunks_text}"
                )
                dpo_pairs.append({
                    "prompt":   shared_prompt,
                    "chosen":   winner["raw_response"],
                    "rejected": loser["raw_response"],
                    "metadata": {
                        "claim_id":          claim_id,
                        "winner_agent":      winner["agent_role"],
                        "loser_agent":       loser["agent_role"],
                        "winner_reward":     brier_reward(winner["confidence_internal"], winner["v_label"]),
                        "loser_reward":      brier_reward(loser["confidence_internal"],  loser["v_label"]),
                        "v_label":           winner["v_label"],
                    }
                })

    # Write outputs
    with open(output_path.replace(".jsonl", "_grpo.jsonl"), "w") as f:
        for r in grpo_records:
            f.write(json.dumps(r) + "\n")

    with open(output_path.replace(".jsonl", "_sft.jsonl"), "w") as f:
        for r in sft_records:
            f.write(json.dumps(r) + "\n")

    with open(output_path.replace(".jsonl", "_dpo.jsonl"), "w") as f:
        for r in dpo_pairs:
            f.write(json.dumps(r) + "\n")

    print(f"✓ GRPO export complete")
    print(f"  grpo records (all outputs): {len(grpo_records)}")
    print(f"  sft records (high quality): {len(sft_records)}")
    print(f"  dpo pairs (winner/loser):   {len(dpo_pairs)}")
    print(f"  Note: rewards are None until Stage 2 (judge) runs.")
    print(f"  Re-run export after judge to get filled Brier rewards.")

# Run export
export_grpo_dataset(DB_PATH, "/content/mad_v2_export.jsonl")

# Download all files
from google.colab import files
for fname in ["/content/mad_v2_export_grpo.jsonl",
              "/content/mad_v2_export_sft.jsonl",
              "/content/mad_v2_export_dpo.jsonl"]:
    try:
        files.download(fname)
    except Exception:
        pass